# DicomStructureFile Relations

This notebook is used for prototyping and testing the `DicomStructureFile` class, which is responsible for handling DICOM RT Structure files in medical imaging applications.



## 1. Import Required Libraries

First, let's import the necessary libraries including our custom DicomStructureFile class.

In [2]:
# Import required libraries
from typing import List, Tuple
#import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import shapely

# Add the src directory to the Python path
#sys.path.append('src')

# Import our custom DicomStructureFile class
from dicom import DicomStructureFile

# Import related classes
from types_and_classes import SliceIndexType
from contours import ContourPoints
from structure_set import StructureSet
from relations import RelationshipType

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
%matplotlib inline

## Demonstrate errors in contours produced by rounding

In [4]:
def points_to_polygon(points: List[Tuple[float, float]]) -> shapely.Polygon:
    '''The points_to_polygon function without validation testing'''
    if not points:
        return shapely.Polygon()
    polygon = shapely.Polygon(points)
    return polygon


# Define the path to the test DICOM file
tests_dir = Path.cwd() / r'Tests'
tests_dir = tests_dir.resolve()
test_file_name = 'RS.GJS_Struct_Tests.BRBL BH.dcm'

print(f"=== Loading DICOM file ===: {test_file_name}")
dicom_file = DicomStructureFile(
    top_dir=tests_dir,
    file_name=test_file_name
)
filtered_contours = dicom_file.filter_exclusions()

dicom_file.structure_names
slice_data = dicom_file.contour_points
contour_table = pd.DataFrame(slice_data)

structure_names_dict = dicom_file.structure_names
structure_names = pd.Series(structure_names_dict, name='StructureName')
structure_names.index.name = 'ROI'
contour_table = contour_table.join(structure_names, on='ROI', how='left')

# Round contour points
dicom_file.round_contour_points()

# Create polygons and calculate areas
contour_table['Polygon'] = contour_table['Points'].apply(points_to_polygon)
contour_table['Area'] = contour_table['Polygon'].apply(lambda poly: poly.area)

# Identify invalid contours
contour_table['IsValid'] = contour_table['Polygon'].apply(shapely.is_valid_reason)
valid_contours = contour_table.IsValid.str.contains('Valid')
selected_columns = ['StructureName', 'ROI', 'Slice', 'IsValid', 'Points']
invalid_contours = contour_table.loc[~valid_contours,selected_columns]
invalid_contours.sort_values(by=['IsValid', 'StructureName'], inplace=True)
print('\nContour Errors:')
print(invalid_contours[['StructureName', 'Slice', 'IsValid']])

print(invalid_contours[invalid_contours.Slice==0.0].Points.tolist())

INFO:dicom:Successfully loaded DICOM dataset from RS.GJS_Struct_Tests.BRBL BH.dcm


=== Loading DICOM file ===: RS.GJS_Struct_Tests.BRBL BH.dcm


INFO:dicom:Extracted 626 contours from 10 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.GJS_Struct_Tests.BRBL BH.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Filtered 0 contours from 1 excluded ROIs. Remaining: 626 contours from 9 ROIs
INFO:dicom:Rounded 626 contours to resolution of 0.1 cm/pixel



Contour Errors:
Empty DataFrame
Columns: [StructureName, Slice, IsValid]
Index: []
[]


# Incorrect relations?


In [5]:
test_file_name = 'RS.GJS_Struct_Tests.BRBL BH.dcm'

print(f"=== Loading DICOM file ===: {test_file_name}")
dicom_file = DicomStructureFile(
    top_dir=tests_dir,
    file_name=test_file_name
)
print(f"Successfully loaded: {dicom_file}")
print("=== Creating StructureSet from DicomStructureFile ===")
structure_set = StructureSet(dicom_structure_file=dicom_file)
print("\n=== Calculating Structure Relationships ===")
structure_set.calculate_relationships()
print("\n=== Structure Relationship Summary Table ===")
symbol_map = {RelationshipType.UNKNOWN: '?',
              RelationshipType.DISJOINT: '',
              RelationshipType.EQUALS: '='
        }
relationship_summary = structure_set.relationship_summary(symbol_map=symbol_map)
#print(relationship_summary)
relationship_summary

INFO:dicom:Successfully loaded DICOM dataset from RS.GJS_Struct_Tests.BRBL BH.dcm


=== Loading DICOM file ===: RS.GJS_Struct_Tests.BRBL BH.dcm


INFO:dicom:Extracted 626 contours from 10 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.GJS_Struct_Tests.BRBL BH.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:structure_set:Building StructureSet from 626 contour points


Successfully loaded: DICOM RT Structure: BRBL BH for patient GJS_Struct_Tests
=== Creating StructureSet from DicomStructureFile ===

=== Calculating Structure Relationships ===


d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: invalid value encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)



=== Structure Relationship Summary Table ===


Structure_B,BODY,Cavity,CTV Cavity,eval PTV,PTV Cavity,Heart,Lung B,Lung L,Lung R
Structure_A,,,,,,,,,
BODY,=,Contains,Contains,Contains,Overlaps,Contains,Partition,Partition,Partition
Cavity,?,=,Overlaps,Overlaps,Overlaps,,,,
CTV Cavity,?,Overlaps,=,Overlaps,Overlaps,,,,
eval PTV,?,Overlaps,Overlaps,=,Overlaps,,,,
PTV Cavity,Overlaps,Overlaps,Overlaps,Overlaps,=,,,,
Heart,?,,,,,=,Borders,Borders,Borders
Lung B,?,,,,,Borders,=,Overlaps,Partition
Lung L,?,,,,,Borders,Overlaps,=,
Lung R,?,,,,,Borders,?,,=


In [18]:
structure_names

14    PTV Cavity
9           BODY
10           DPV
20        Lung R
13      eval PTV
12    CTV Cavity
11        Cavity
19        Lung L
18        Lung B
17         Heart
dtype: object

In [ ]:

contour_list = pd.DataFrame(dicom_file.contour_points)
structure_names = pd.Series(dicom_file.structure_names, name='StructureName')
contours = contour_list.join(structure_names, on='ROI', how='left')

contours = contours.groupby(['StructureName', 'ROI', 'Slice']).count()
slice_table = contours.unstack(level=['StructureName', 'ROI'])
slice_table.columns = slice_table.columns.droplevel(0)
slice_table.loc[1.0,:].to_dict()

{('BODY', 9): 1.0,
 ('CTV Cavity', 12): 1.0,
 ('Cavity', 11): 1.0,
 ('Heart', 17): nan,
 ('Lung B', 18): 2.0,
 ('Lung L', 19): 1.0,
 ('Lung R', 20): 1.0,
 ('PTV Cavity', 14): 1.0,
 ('eval PTV', 13): 1.0}

In [ ]:
from contours import build_contour_table
from debug_tools import plot_roi

contour_list = pd.DataFrame(dicom_file.contour_points)
structure_names = pd.Series(dicom_file.structure_names, name='StructureName')

contour_table, slice_sequence = build_contour_table(contour_list)
contour_table = contour_table.join(structure_names, on='ROI', how='left')

def combine_polygons(polygons):
    valid_polygons = []
    for poly in polygons:
        if isinstance(poly, shapely.geometry.Polygon):
            if poly.is_valid:
                valid_polygons.append(poly)
    if not valid_polygons:
        return None
    return shapely.union_all(valid_polygons)
slice_group = contour_table.groupby(['StructureName', 'ROI', 'Slice'])
slice_table = slice_group.aggregate(combine_polygons)
slice_table.drop(columns=['Points', 'Area'], inplace=True)
slice_table = slice_table.unstack(level=['StructureName', 'ROI'])
slice_table.columns = slice_table.columns.droplevel(0)

In [56]:
slice_table

StructureName,BODY,CTV Cavity,Cavity,Heart,Lung B,Lung L,Lung R,PTV Cavity,eval PTV
ROI,9,12,11,17,18,19,20,14,13
Slice,,,,,,,,,
-13.75,MULTIPOLYGON Z (((-2.3649999999999998 -19.072 ...,NaN,NaN,NaN,"MULTIPOLYGON Z (((-15.013 -2.393 -13.75, -15.0...","POLYGON Z ((17.022 -1.339 -13.75, 17.062 -1.29...",POLYGON Z ((-14.971 -2.4379999999999997 -13.75...,NaN,NaN
-13.50,"MULTIPOLYGON Z (((-6.006 -18.799 -13.5, -6.221...",NaN,NaN,NaN,"MULTIPOLYGON Z (((-14.825 -2.939 -13.5, -14.94...","POLYGON Z ((16.747999999999998 -1.52 -13.5, 17...","POLYGON Z ((-14.697 -3.06 -13.5, -14.568999999...",NaN,NaN
-13.25,"MULTIPOLYGON Z (((-5.732 -18.799 -13.25, -5.94...",NaN,NaN,NaN,MULTIPOLYGON Z (((-14.902000000000001 -3.213 -...,POLYGON Z ((16.747999999999998 -1.835999999999...,"POLYGON Z ((-14.697 -3.444 -13.25, -14.4340000...",NaN,NaN
-13.00,"MULTIPOLYGON Z (((-5.225 -18.799 -13, -5.4 -18...",NaN,NaN,NaN,MULTIPOLYGON Z (((-14.841999999999999 -4.03299...,POLYGON Z ((16.747999999999998 -2.401000000000...,"POLYGON Z ((-14.697 -4.25 -13, -14.42400000000...",NaN,NaN
-12.75,"MULTIPOLYGON Z (((-4.922 -18.799 -12.75, -5.12...",NaN,NaN,NaN,MULTIPOLYGON Z (((-14.613999999999999 -4.854 -...,"POLYGON Z ((17.022 -4.188000000000001 -12.75, ...","POLYGON Z ((-14.424000000000001 -5.116 -12.75,...",NaN,NaN
...,...,...,...,...,...,...,...,...,...
13.00,"POLYGON Z ((1.436 -11.456 13, 1.709 -11.442 13...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13.25,"POLYGON Z ((1.162 -11.797 13.25, 1.436 -11.783...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
plot_roi(contour_table, [11, 12])